Projeler: Senin listenden 7 tane:

Yapay Sinir Ağları ile Görüntü Sınıflandırma (MLP) -yapmıştım tekrar yapmicam

CNN ile Rakam Görüntüsü Sınıflandırma (MNIST) -yapmıştım tekrar yapmicam

LSTM ile Film Yorumları Duygu Analizi

GAN ile Görüntü Oluşturma

Transformers ile Duygu Analizi (Text)

Autoencoder ile Moda Görüntüleri (Fashion-MNIST)

Transfer Learning ile İlaç Görüntüleri Sınıflandırma

**Transformers ile Duygu Analizi (Text)**

In [3]:
!pip install transformers datasets torch -q  # transformers (BERT vb.), datasets (hazır veri setleri), torch (deep learning kütüphanesi) yüklenir, -q sessiz kurulum yapar

import torch  # PyTorch kütüphanesi; tensor işlemleri ve model eğitimi için kullanılır
from datasets import load_dataset  # HuggingFace datasets kütüphanesinden veri seti yüklemek için kullanılır
from transformers import AutoTokenizer  # Metni sayısal hale getiren tokenizer'ı otomatik seçmek için kullanılır
from transformers import AutoModelForSequenceClassification  # Hazır eğitilmiş sınıflandırma modeli (BERT vb.) yüklemek için kullanılır
from transformers import TrainingArguments, Trainer  # Model eğitimini kolaylaştıran hazır eğitim araçları

In [4]:
dataset = load_dataset("imdb")  # HuggingFace üzerinden IMDb film yorumları veri setini indirir ve dataset değişkenine atar

print(dataset)  # Dataset'in genel yapısını (train/test bölümleri, veri sayıları) ekrana yazdırır

print(dataset["train"][0])  # Eğitim verisindeki ilk örneği gösterir (text + label)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [5]:
for i in range(5):  # 0'dan 4'e kadar ilk 5 veriyi döner
    print(dataset["train"][i])  # her bir veriyi ekrana yazdırır
    print("------")  # verileri ayırmak için çizgi koyar

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")  # hazır BERT tokenizer'ını indirir (küçük harfli İngilizce model)

def tokenize_function(example):  # her veri örneği için çalışacak fonksiyonu tanımlar
    return tokenizer(
        example["text"],  # text alanını alır (film yorumu)
        padding="max_length",  # tüm verileri aynı uzunluğa getirir (eksik olanları doldurur)
        truncation=True,  # çok uzun metinleri keser (modelin kabul ettiği max uzunluğa indirir)
        max_length=256  # maksimum 256 token uzunluk belirler (çok uzunları keser)
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)  # tüm dataset'e tokenize işlemini uygular (batch halinde daha hızlı)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",  # hazır BERT modeli (küçük harfli İngilizce)
    num_labels=2  # sınıf sayısı (pozitif / negatif → 2 sınıf)
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
training_args = TrainingArguments(
    output_dir="./results",  # model çıktılarının kaydedileceği klasör (checkpoint vs)
    eval_strategy="epoch",  # HER EPOCH sonunda modeli test et (evaluation_strategy yerine bu kullanılıyor)
    learning_rate=2e-5,  # öğrenme hızı → model ağırlıkları ne kadar değişsin
    per_device_train_batch_size=8,  # her adımda 8 veri ile öğrenir (RAM dengesi için düşük tutulur)
    per_device_eval_batch_size=8,  # test sırasında da 8 veri kullanılır
    num_train_epochs=2,  # tüm veriyi 2 kez görür (epoch sayısı)
    weight_decay=0.01,  # aşırı öğrenmeyi (overfitting) azaltır
)

In [9]:
trainer = Trainer(
    model=model,  # eğitilecek BERT modeli (duygu analizi için)
    args=training_args,  # biraz önce tanımladığımız eğitim ayarları
    train_dataset=tokenized_datasets["train"],  # modelin öğreneceği veri (25k yorum)
    eval_dataset=tokenized_datasets["test"],  # modelin test edileceği veri (25k yorum)
)

In [10]:
trainer.train()  # modeli eğitmeye başlatır (en önemli satır)

Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
trainer.evaluate()  # modelin test verisindeki performansını ölçer

In [ ]:
text = "This movie was absolutely amazing!"  # modelin yorumlayacağı yeni bir cümle

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)  # metni modele uygun tensor formatına çevirir

outputs = model(**inputs)  # modeli çalıştırır ve tahmin üretir

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)  # çıktıyı olasılıklara çevirir

predicted_class = torch.argmax(predictions).item()  # en yüksek olasılığa sahip sınıfı seçer

print(predicted_class)  # sonucu yazdırır (0 negatif, 1 pozitif)

In [ ]:
from sklearn.metrics import accuracy_score  # modelin doğruluk oranını hesaplamak için gerekli fonksiyon

def compute_metrics(eval_pred):  # Trainer'ın evaluation sırasında çağıracağı fonksiyon
    logits, labels = eval_pred  # modelin çıktıları (logits) ve gerçek etiketler (labels) alınır
    predictions = logits.argmax(axis=-1)  # en yüksek skora sahip sınıf seçilir (0 veya 1)
    acc = accuracy_score(labels, predictions)  # gerçek ve tahmin karşılaştırılır → accuracy hesaplanır
    return {"accuracy": acc}  # Trainer'ın anlayacağı formatta sözlük döndürülür

In [ ]:
trainer = Trainer(
    model=model,  # eğitilecek model
    args=training_args,  # eğitim ayarları
    train_dataset=tokenized_datasets["train"],  # eğitim verisi
    eval_dataset=tokenized_datasets["test"],  # test verisi
    compute_metrics=compute_metrics,  # az önce yazdığımız accuracy fonksiyonunu ekler
)

In [ ]:
trainer.train()  # modeli eğitir

results = trainer.evaluate()  # test verisinde performansı ölçer

print(results)  # accuracy ve loss değerlerini ekrana yazdırır

In [ ]:
from sklearn.metrics import confusion_matrix  # confusion matrix hesaplamak için
import numpy as np  # array işlemleri için

predictions = trainer.predict(tokenized_datasets["test"])  # test verisi için tahmin yapar

preds = np.argmax(predictions.predictions, axis=-1)  # en yüksek olasılıklı sınıfı seçer
labels = predictions.label_ids  # gerçek etiketleri alır

cm = confusion_matrix(labels, preds)  # confusion matrix oluşturur

print(cm)  # matrisi yazdırır

In [11]:
import matplotlib.pyplot as plt  # grafik çizmek için

accuracy = results["eval_accuracy"]  # accuracy değerini alır

plt.bar(["Accuracy"], [accuracy])  # tek bar grafik oluşturur
plt.title("Model Accuracy")  # grafik başlığı
plt.show()  # grafiği ekranda gösterir

NameError: name 'results' is not defined